# Chapter 9 — Hello, GPU

> Course: **llm.c — Zero to Hero**, Chapter 9 of ~20.  **Start of Part II.**

Welcome to Part II. Everything you've built so far runs on a CPU. Now we pivot.

GPUs are not "fast CPUs". They're a fundamentally different kind of processor — built for **massive data-parallel** work: thousands of small, identical computations running concurrently. A modern GPU has tens of thousands of arithmetic units, vs. dozens of cores on your CPU. To use them you don't write multi-threaded code with `pthread` or `#pragma omp`; you write **kernels** — small functions that the runtime launches across a grid of threads.

This chapter is the gentlest possible introduction. We'll write **two** CUDA programs: a `hello_world` that prints from each thread, and a port of `gelu_forward` (Chapter 5) onto the GPU. Both are runnable, both demonstrate the *bare minimum* you need to know about CUDA.

### Learning objectives

By the end of this chapter you will:

- Write a CUDA kernel using `__global__`, identify a thread by `threadIdx.x` and `blockIdx.x`, and launch it with `<<<grid, block>>>` syntax.
- Allocate device memory with `cudaMalloc`, copy host↔device with `cudaMemcpy`, free with `cudaFree`.
- Explain the **host vs device** split: which code runs on the CPU, which on the GPU.
- Port a CPU loop to a CUDA kernel — your first GPU port of an `llm.c` layer.


## 1. Concept — The CUDA Programming Model in 60 Seconds

A CUDA program is a normal C program with two extra ideas:

1. **Kernel functions** marked `__global__`. They run on the GPU. They're called from CPU code with the syntax `kernel_name<<<gridDim, blockDim>>>(args)`.
2. **Threads, blocks, and grids.** When you launch a kernel, the runtime spawns `gridDim × blockDim` **threads**, *all running the same kernel code simultaneously*. Each thread can read its **own** identity from built-in variables `threadIdx`, `blockIdx`, `blockDim`, `gridDim` — that's how a thread knows *"which element of the array am I responsible for?"*.

The hierarchy:

```
grid    = collection of blocks
block   = collection of threads (typically 32 - 1024)
thread  = the smallest execution unit, runs the kernel code
```

Each thread computes its **global linear index** as:

```c
int i = blockIdx.x * blockDim.x + threadIdx.x;
```

That `i` is the thread's "address". For most simple kernels (like GELU), thread `i` reads `inp[i]`, computes something, and writes `out[i]`. **One thread per element.**

A modern GPU executes 32 threads (a *warp*) in lockstep, hundreds of warps per block, thousands of blocks across the grid. A single `<<<>>>` launch is launching **millions of threads at once**.


Here's the picture to hold in your head (from *CUDA by Example*, Fig. 5.2). A **grid** is an array of **blocks**; each block is an array of **threads**. The green zoom expands one block — `Block (0,1)` — into its 16×16 threads, each labelled by its `(threadIdx.x, threadIdx.y)`:

![A grid of blocks, with one block expanded into its threads](course/figures/fig_5_2_block_thread_hierarchy.png)

*The grid here is 3×2 blocks, each block 16×16 threads. A thread is addressed in two levels: **which block** (`blockIdx`) and **which thread inside that block** (`threadIdx`).*

In **this** chapter we use only the **1D slice** of this picture — just the `.x` axis (`blockIdx.x`, `blockDim.x`, `threadIdx.x`), so mentally collapse the figure to a single row of blocks, each a single row of threads. The full 2D form (used for images and matrices) returns in the 2D-grids companion notebook. Read the global-index formula off the picture: `blockIdx.x * blockDim.x + threadIdx.x` is exactly *"skip all the threads in the blocks before me (`blockIdx.x` blocks × `blockDim.x` threads each), then add my offset within my own block (`threadIdx.x`)."*


## 2. Concept — Host vs Device Memory

CPU memory (RAM) and GPU memory (VRAM) are **separate**. A pointer that's valid on the CPU isn't valid on the GPU and vice versa. The CUDA runtime gives you four functions to manage this:

| Function | What it does |
|---|---|
| `cudaMalloc(&p, bytes)` | Allocate `bytes` of device memory; `p` is now a device pointer |
| `cudaMemcpy(dst, src, bytes, dir)` | Copy host↔device. `dir` is `cudaMemcpyHostToDevice` or `cudaMemcpyDeviceToHost` |
| `cudaFree(p)` | Free device memory |
| `cudaDeviceSynchronize()` | Block until all queued kernels finish (otherwise launches are async) |

Mental model: the CPU and GPU are **two different machines linked by PCIe**. Every CUDA program does:

1. CPU allocates host buffers, fills them with data.
2. CPU `cudaMalloc`s device buffers.
3. CPU `cudaMemcpy`s data from host → device.
4. CPU launches a kernel; thousands of GPU threads run on the device buffers.
5. CPU `cudaMemcpy`s results from device → host.
6. CPU does whatever it needs with the results, then `cudaFree`s the device buffers.

Steps 3 and 5 are the **expensive** parts (PCIe is much slower than VRAM). Real production code minimizes the number of these transfers — keep data on the GPU as long as possible.


## 3. PyTorch Baseline

You've been doing this all along without thinking about it:

```python
x = torch.randn(1000)              # on CPU
x_gpu = x.cuda()                    # cudaMemcpy(host -> device)
y_gpu = F.gelu(x_gpu, approximate='tanh')   # launches a CUDA kernel
y = y_gpu.cpu()                     # cudaMemcpy(device -> host)
```

PyTorch hides every CUDA call. We're about to write the same five-step dance by hand.


## 4. Hello, GPU — Your First Kernel

In [1]:
!mkdir -p course/ch09_build


In [2]:
%%writefile course/ch09_build/hello_gpu.cu
#include <stdio.h>
#include <cuda_runtime.h>

// __global__ marks this as a kernel: it runs on the GPU,
// but is callable from CPU code via the <<<...>>> syntax.
__global__ void hello_kernel(void) {
    // Each thread computes its global index from the launch geometry.
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Hello from block %d thread %d (global id %d)\n",
           blockIdx.x, threadIdx.x, tid);
}

int main(void) {
    // Launch with 2 blocks, 4 threads each = 8 threads total.
    // Syntax: kernel<<<gridDim, blockDim>>>(args);
    hello_kernel<<<2, 4>>>();

    // The launch is asynchronous; wait for the GPU to finish before exiting.
    cudaDeviceSynchronize();
    return 0;
}


Overwriting course/ch09_build/hello_gpu.cu


In [5]:
!nvcc -O2 -o course/ch09_build/hello_gpu course/ch09_build/hello_gpu.cu && ./course/ch09_build/hello_gpu


Hello from block 1 thread 0 (global id 4)
Hello from block 1 thread 1 (global id 5)
Hello from block 1 thread 2 (global id 6)
Hello from block 1 thread 3 (global id 7)
Hello from block 0 thread 0 (global id 0)
Hello from block 0 thread 1 (global id 1)
Hello from block 0 thread 2 (global id 2)
Hello from block 0 thread 3 (global id 3)


You should see 8 lines of "Hello from..." output. The lines may be **out of order** — that's not a bug! Threads execute concurrently and `printf` from multiple threads gets interleaved nondeterministically.

Three things to internalize from this:

1. **`__global__`** marks GPU code. The function is *called* from the host (CPU) but *runs* on the device (GPU).
2. **`<<<2, 4>>>`** is the launch geometry: 2 blocks, 4 threads per block.
3. **`threadIdx.x`** is `0..3` within each block; **`blockIdx.x`** is `0` or `1`. The global thread id `blockIdx.x * blockDim.x + threadIdx.x` runs from `0..7`.

That's the *entire* CUDA programming model. Everything else is variations on this theme.


## 5. GELU on the GPU — One Thread per Element

Now let's actually do work. We'll port `gelu_forward` from Chapter 5 to a CUDA kernel.

Compare the structure side by side. The CPU version (Chapter 5):

```c
void gelu_forward_cpu(float* out, const float* inp, int N) {
    for (int i = 0; i < N; i++) {                            // <-- loop over N elements
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        out[i] = 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
    }
}
```

The GPU version (from `dev/cuda/gelu_forward.cu` kernel 1):

```c
__global__ void gelu_forward_kernel1(float* out, const float* inp, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;            // <-- "loop body" with i precomputed
    if (i < N) {                                              // <-- bounds check (no `for` loop!)
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        out[i] = 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
    }
}
```

**The only difference is that the `for (int i = 0; i < N; i++)` loop is replaced by computing `i` from the thread coordinates.** The body — the GELU math — is **byte-identical**. The CPU does one iteration of the loop sequentially per cycle; the GPU launches `N` threads, each doing one iteration in parallel.

That's the whole port.


## 6. The Five-Step Dance — Allocate, Copy, Launch, Copy, Free

In [6]:
%%writefile course/ch09_build/gelu_forward.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

// ----- CPU reference (so we can verify) -----
void gelu_forward_cpu(float* out, const float* inp, int N) {
    for (int i = 0; i < N; i++) {
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        out[i] = 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
    }
}

// ----- GPU kernel: one thread per element -----
__global__ void gelu_forward_kernel1(float* out, const float* inp, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {                                      // guard: don't run off the end
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        out[i] = 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
    }
}

int main(int argc, char** argv) {
    int N = (argc == 2) ? atoi(argv[1]) : 1024;

    // 1. allocate HOST buffers, fill input
    float* h_inp = (float*) malloc(N * sizeof(float));
    float* h_out = (float*) malloc(N * sizeof(float));
    float* h_ref = (float*) malloc(N * sizeof(float));
    for (int i = 0; i < N; i++) h_inp[i] = (float)(i - N/2) * 0.01f;  // a range around 0

    // CPU reference
    gelu_forward_cpu(h_ref, h_inp, N);

    // 2. allocate DEVICE buffers
    float *d_inp, *d_out;
    cudaMalloc(&d_inp, N * sizeof(float));
    cudaMalloc(&d_out, N * sizeof(float));

    // 3. copy input host -> device
    cudaMemcpy(d_inp, h_inp, N * sizeof(float), cudaMemcpyHostToDevice);

    // 4. launch the kernel
    int block_size = 256;                                  // threads per block
    int grid_size  = (N + block_size - 1) / block_size;    // ceil(N / block_size)
    gelu_forward_kernel1<<<grid_size, block_size>>>(d_out, d_inp, N);

    // 5. copy result device -> host
    cudaMemcpy(h_out, d_out, N * sizeof(float), cudaMemcpyDeviceToHost);

    // verify
    float maxerr = 0;
    for (int i = 0; i < N; i++) {
        float e = fabsf(h_out[i] - h_ref[i]);
        if (e > maxerr) maxerr = e;
    }
    printf("N=%d  block_size=%d  grid_size=%d  max |gpu - cpu| = %.2e\n",
           N, block_size, grid_size, maxerr);

    cudaFree(d_inp);
    cudaFree(d_out);
    free(h_inp); free(h_out); free(h_ref);
    return 0;
}


Overwriting course/ch09_build/gelu_forward.cu


In [7]:
!nvcc -O2 -o course/ch09_build/gelu_forward course/ch09_build/gelu_forward.cu && ./course/ch09_build/gelu_forward 1024


N=1024  block_size=256  grid_size=4  max |gpu - cpu| = 2.38e-07


In [8]:
# Try a few sizes — note that the kernel handles N that isn't a multiple of block_size
!./course/ch09_build/gelu_forward 100
!./course/ch09_build/gelu_forward 1000000


N=100  block_size=256  grid_size=1  max |gpu - cpu| = 2.98e-08
N=1000000  block_size=256  grid_size=3907  max |gpu - cpu| = 2.38e-07


All three runs print a **tiny** `max |gpu - cpu|` on the order of `1e-7` (we measured `2.38e-07`), **not** exactly zero. The formula is the same single-precision arithmetic, but the GPU evaluates `tanhf` with its own device intrinsic, which differs from your host C library's `tanhf` by about 1 ULP. That ~`1e-7` gap is transcendental rounding, not a bug — for `float` it's the expected level of host↔device agreement. (A kernel using only `+`, `-`, `*` — like the `residual` exercise below — *does* match bit-for-bit, because integer/exact-float ops are identical on both.)

Note how `grid_size = ceil(N / block_size)` accommodates any `N`: for `N = 100`, block_size 256 → grid_size 1, so we launch 256 threads but only the first 100 do work (the `if (i < N)` guard kills the rest).


## 7. The Translation Bridge

| OpenMP CPU | CUDA GPU |
|---|---|
| `for (int i = 0; i < N; i++) body(i);` | `__global__ kernel(...) { int i = blockIdx.x * blockDim.x + threadIdx.x; if (i < N) body(i); }` |
| `#pragma omp parallel for` | `kernel<<<grid_size, block_size>>>(...)` |
| `malloc`, `free` | `cudaMalloc`, `cudaFree` |
| `memcpy` (within RAM) | `cudaMemcpy(..., cudaMemcpyHostToDevice / DeviceToHost)` |
| Threads share RAM | Host has RAM; device has VRAM. They are *separate*. |
| ~28 cores on a workstation CPU | ~10,000+ CUDA cores on an RTX 4080 |

Mental model: **a CUDA kernel is the body of a `for` loop, with the loop variable replaced by the thread's global index.** That's the whole abstraction. Everything in Chapters 10-19 is making this go faster: choosing a good `block_size`, processing multiple elements per thread, using shared memory to cache reads, doing tree reductions for sums and maxes, etc. But the foundation is what you just wrote.


## 8. Choosing `block_size`

The `<<<grid, block>>>` numbers aren't free. Some rules of thumb:

- **`block_size` must be a multiple of 32** (the warp size). Anything else wastes hardware.
- **`block_size` between 128 and 1024.** Below 128, you can't keep all the schedulers busy. Above 1024 is illegal (hardware limit).
- **Common defaults: 128, 256, 512.** `llm.c` uses 256–512 most often.
- **The runtime tunes automatically for many cases.** Try a few and benchmark.

Let's see how `block_size` affects throughput on a large GELU.


In [9]:
%%writefile course/ch09_build/gelu_bench.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define GELU_SCALING_FACTOR sqrtf(2.0f / M_PI)

__global__ void gelu_kernel(float* out, const float* inp, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        float x = inp[i];
        float cube = 0.044715f * x * x * x;
        out[i] = 0.5f * x * (1.0f + tanhf(GELU_SCALING_FACTOR * (x + cube)));
    }
}

int main(void) {
    const int N = 1 << 24;     // ~16M elements (~64 MB)
    float *d_inp, *d_out;
    cudaMalloc(&d_inp, N * sizeof(float));
    cudaMalloc(&d_out, N * sizeof(float));

    int block_sizes[] = {32, 64, 128, 256, 512, 1024};
    int n_bs = 6;
    int iters = 50;

    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);

    printf("N = %d (%.1f MB)\n", N, N*4.0/(1024*1024));
    printf("block_size  grid_size   ms/iter   bandwidth (GB/s)\n");
    for (int b = 0; b < n_bs; b++) {
        int block_size = block_sizes[b];
        int grid_size  = (N + block_size - 1) / block_size;

        // warmup
        gelu_kernel<<<grid_size, block_size>>>(d_out, d_inp, N);
        cudaDeviceSynchronize();

        cudaEventRecord(start);
        for (int k = 0; k < iters; k++)
            gelu_kernel<<<grid_size, block_size>>>(d_out, d_inp, N);
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float ms;
        cudaEventElapsedTime(&ms, start, stop);
        float per_iter = ms / iters;
        // 2 floats touched per element (read + write), 4 bytes each
        float bw_gb_s = (2.0f * N * 4.0f / 1e9f) / (per_iter / 1000.0f);
        printf("%9d  %9d  %8.3f  %14.1f\n", block_size, grid_size, per_iter, bw_gb_s);
    }

    cudaFree(d_inp); cudaFree(d_out);
    return 0;
}


Overwriting course/ch09_build/gelu_bench.cu


In [10]:
!nvcc -O2 -o course/ch09_build/gelu_bench course/ch09_build/gelu_bench.cu && ./course/ch09_build/gelu_bench


N = 16777216 (64.0 MB)
block_size  grid_size   ms/iter   bandwidth (GB/s)
       32     524288     0.257           522.2
       64     262144     0.213           629.3
      128     131072     0.214           628.0
      256      65536     0.214           627.9
      512      32768     0.214           626.6
     1024      16384     0.204           659.4


On most modern GPUs, **block_size 64–1024 land within ~15% of each other**, with `128–512` a safe default. `block_size 32` is the slowest (one warp per block gives the scheduler less latency to hide) — on a fast GPU like this the gap is modest, maybe 10–20%, not catastrophic; on smaller or older GPUs it hurts more. Very large blocks (`1024`) aren't always optimal either, since register pressure can cap occupancy.

The "bandwidth (GB/s)" column tells you what's actually happening: GELU is **bandwidth-bound** — it does 5 FLOPs per element but moves 8 bytes (read + write). Modern GPUs have 800+ GB/s of memory bandwidth; you should see most of it. If you see numbers far below your GPU's peak, try a larger `N` or a different `block_size`.


## 9. Common Pitfalls (Read Before Writing Any More CUDA)

Three rookie mistakes to avoid forever:

1. **Forgetting `cudaDeviceSynchronize()` before timing.**
   Kernel launches return immediately to the host; the GPU runs asynchronously. If you time only the launch call, you measure ~10 microseconds of host-side dispatch, not the kernel. The benchmark above uses `cudaEvent` which records *on the GPU's timeline* and is the right way.

2. **Forgetting the `if (i < N)` bounds check.**
   `grid_size = (N + block_size - 1) / block_size` rounds *up*. The last block may have threads with `i >= N`. Without the check, those threads write past your buffer — the classic CUDA out-of-bounds bug, often producing silent corruption rather than a clean crash.

3. **Treating CPU and GPU pointers interchangeably.**
   Calling `printf("%f\n", d_out[0])` from host code dereferences a *device* pointer on the host — segfault. You must `cudaMemcpy` first. (Some setups support unified memory, but for `llm.c` we always allocate explicit device buffers.)

The repo's `cuda_check` macro wraps every CUDA call to detect errors; we'll start using it from Chapter 10.


## 10. TODO Exercise — Port `residual_forward` to CUDA

The simplest `llm.c` layer is `residual_forward(out, inp1, inp2, N) { for (i) out[i] = inp1[i] + inp2[i]; }` (Chapter 5). Port it to a CUDA kernel.


In [ ]:
%%writefile course/ch09_build/exercise1.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// TODO: write a CUDA kernel that does out[i] = inp1[i] + inp2[i] for one element.
__global__ void residual_kernel(float* out, const float* inp1, const float* inp2, int N) {
    // 1. compute global index i
    // 2. if i < N, do out[i] = inp1[i] + inp2[i]
}

int main(void) {
    const int N = 4096;
    float *h_a = (float*) malloc(N*sizeof(float));
    float *h_b = (float*) malloc(N*sizeof(float));
    float *h_o = (float*) malloc(N*sizeof(float));
    for (int i = 0; i < N; i++) { h_a[i] = (float)i; h_b[i] = (float)(2*i); }

    float *d_a, *d_b, *d_o;
    cudaMalloc(&d_a, N*4); cudaMalloc(&d_b, N*4); cudaMalloc(&d_o, N*4);
    cudaMemcpy(d_a, h_a, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, N*4, cudaMemcpyHostToDevice);

    // TODO: pick block_size 256, compute grid_size = ceil(N/block_size), launch kernel.
    // <your kernel launch here>

    cudaMemcpy(h_o, d_o, N*4, cudaMemcpyDeviceToHost);

    int correct = 1;
    for (int i = 0; i < N; i++) if (h_o[i] != 3*i) { correct = 0; break; }
    printf("%s  (h_o[0..3] = %.0f %.0f %.0f %.0f, expected 0 3 6 9)\n",
           correct ? "PASS" : "FAIL", h_o[0], h_o[1], h_o[2], h_o[3]);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_o);
    free(h_a); free(h_b); free(h_o);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch09_build/exercise1 course/ch09_build/exercise1.cu && ./course/ch09_build/exercise1


### Solution to Exercise

In [ ]:
%%writefile course/ch09_build/exercise1_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void residual_kernel(float* out, const float* inp1, const float* inp2, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) out[i] = inp1[i] + inp2[i];
}

int main(void) {
    const int N = 4096;
    float *h_a = (float*) malloc(N*sizeof(float));
    float *h_b = (float*) malloc(N*sizeof(float));
    float *h_o = (float*) malloc(N*sizeof(float));
    for (int i = 0; i < N; i++) { h_a[i] = (float)i; h_b[i] = (float)(2*i); }

    float *d_a, *d_b, *d_o;
    cudaMalloc(&d_a, N*4); cudaMalloc(&d_b, N*4); cudaMalloc(&d_o, N*4);
    cudaMemcpy(d_a, h_a, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, N*4, cudaMemcpyHostToDevice);

    int block_size = 256;
    int grid_size = (N + block_size - 1) / block_size;
    residual_kernel<<<grid_size, block_size>>>(d_o, d_a, d_b, N);

    cudaMemcpy(h_o, d_o, N*4, cudaMemcpyDeviceToHost);
    int correct = 1;
    for (int i = 0; i < N; i++) if (h_o[i] != 3*i) { correct = 0; break; }
    printf("%s  (h_o[0..3] = %.0f %.0f %.0f %.0f, expected 0 3 6 9)\n",
           correct ? "PASS" : "FAIL", h_o[0], h_o[1], h_o[2], h_o[3]);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_o);
    free(h_a); free(h_b); free(h_o);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch09_build/exercise1_sol course/ch09_build/exercise1_sol.cu && ./course/ch09_build/exercise1_sol


## Recap

You now know:

- A CUDA kernel is a `__global__` function. It runs on the GPU and looks just like a CPU function, except instead of a `for` loop you have **threads**, each computing its own index from `blockIdx.x * blockDim.x + threadIdx.x`.
- `<<<grid, block>>>` launches the kernel across `grid * block` threads.
- The CPU and GPU have **separate memories**. `cudaMalloc` and `cudaMemcpy` move data between them.
- Always include the `if (i < N)` bounds check; always synchronize before timing.

### What's next

**Chapter 10 — Grid-Stride Loops & Element-wise Ops.** What if `N` is bigger than `gridDim * blockDim`? You add a *grid-stride loop* — the canonical way to write a kernel that's correct for any input size. We'll look at how `llm.c`'s production kernels (`encoder_forward_kernel3`, etc.) use this pattern, and write our own.

When you're ready, say **"proceed to Chapter 10"**.
